In [1]:
# =========================
# 0. Install dependencies
# =========================

!pip install -q transformers peft evaluate tomli scikit-learn pandas tqdm torch


[notice] A new release of pip is available: 23.2.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# =========================
# 1. Imports
# =========================
from pathlib import Path
import os
import re
import json
import sys
import subprocess
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup,
)

C:\Users\aiger\Documents\2026SS\AIR\git_personal_active\AIR_Group_Task\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# =========================
# 2. Paths and config
# =========================
PROJECT_ROOT = Path.cwd()

while not (
    (PROJECT_ROOT / "task2" / "reasoning_trace_build.py").exists()
    and (PROJECT_ROOT / "task2" / "scorer.py").exists()
):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError(
            "Could not find project root. "
            "Run this notebook from inside the cloned repository."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
print("Current directory:", os.getcwd())

RAW_TRAIN_PATH = PROJECT_ROOT / "data" / "english" / "english_train.json"
VAL_PATH = PROJECT_ROOT / "data" / "english" / "clef2026_gpt4_o_mini_val.json"

TRAIN_JSONL = PROJECT_ROOT / "output" / "training_data_for_RM" / "english_train.jsonl"
MODEL_DIR = PROJECT_ROOT / "output" / "baseline_distilbert"
PRED_PATH = PROJECT_ROOT / "output" / "RM_prediction" / "distilbert_predictions.json"
RESULT_DIR = PROJECT_ROOT / "output" / "results_distilbert"

BASE_MODEL = "distilbert-base-uncased"

MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-5
RANDOM_STATE = 42

os.makedirs(PROJECT_ROOT / "output" / "training_data_for_RM", exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PROJECT_ROOT / "output" / "RM_prediction", exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Project root: C:\Users\aiger\Documents\2026SS\AIR\git_personal_active\AIR_Group_Task
Current directory: C:\Users\aiger\Documents\2026SS\AIR\git_personal_active\AIR_Group_Task
Device: cpu


In [9]:
# =========================
# 3. Run provided preprocessing script
# =========================
subprocess.run(
    [
        sys.executable,
        str(PROJECT_ROOT / "task2" / "reasoning_trace_build.py"),
        "--input",
        str(RAW_TRAIN_PATH),
        "--output",
        str(TRAIN_JSONL),
    ],
    check=True,
    cwd=str(PROJECT_ROOT),
)

train_df = pd.read_json(TRAIN_JSONL, lines=True)

print(train_df.head())
print(train_df.columns)
print(train_df["Class"].value_counts())
print("Labels:", train_df["Label"].unique())
print("Verdicts:", train_df["Verdict"].unique())

  sample_id                                         input_text        Label  \
0       0_a  Claim: “The first randomized controlled trial ...  conflicting   
1       0_b  Claim: “The first randomized controlled trial ...  conflicting   
2       0_c  Claim: “The first randomized controlled trial ...  conflicting   
3       0_d  Claim: “The first randomized controlled trial ...  conflicting   
4       0_e  Claim: “The first randomized controlled trial ...  conflicting   

  Verdict  Class  
0   false      0  
1   false      0  
2   false      0  
3   false      0  
4   false      0  
Index(['sample_id', 'input_text', 'Label', 'Verdict', 'Class'], dtype='str')
Class
0    22775
1     8658
Name: count, dtype: int64
Labels: <ArrowStringArray>
['conflicting', 'false', 'true']
Length: 3, dtype: str
Verdicts: <ArrowStringArray>
['false', 'conflicting', 'true']
Length: 3, dtype: str


In [3]:
# =========================
# 4. Dataset
# =========================

class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["input_text"].tolist()
        self.labels = dataframe["Class"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)

        return item

In [4]:
# =========================
# 5. DistilBERT verifier model
# =========================
class CustomClassifier(torch.nn.Module):
    def __init__(
        self,
        model_name,
        num_labels=1,
        hidden_dim=None,
        dropout_value=0.1,
        freeze_base_layer=False,
    ):
        super().__init__()

        self.model = AutoModel.from_pretrained(model_name)

        if freeze_base_layer:
            for param in self.model.parameters():
                param.requires_grad = False

        hidden_size = self.model.config.hidden_size

        if hidden_dim:
            self.classifier = torch.nn.Sequential(
                torch.nn.Linear(hidden_size, hidden_dim),
                torch.nn.ReLU(),
                torch.nn.Dropout(dropout_value),
                torch.nn.Linear(hidden_dim, num_labels),
            )
        else:
            self.classifier = torch.nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)

        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = pooled_output.float()

        logits = self.classifier(pooled_output)
        return logits

In [5]:
# =========================
# 6. Helper function
# =========================

def print_trainable_parameters(model):
    trainable_params = 0
    all_params = 0

    for _, param in model.named_parameters():
        all_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    print(
        f"trainable params: {trainable_params} || "
        f"all params: {all_params} || "
        f"trainable%: {100 * trainable_params / all_params:.2f}"
    )

In [6]:
# =========================
# 7. Trainer
# =========================

class TrainerModule:
    def __init__(
        self,
        model,
        train_loader,
        val_loader,
        epochs,
        lr,
        output_dir,
    ):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)

        self.train_loader = train_loader
        self.val_loader = val_loader
        self.epochs = epochs

        self.optimizer = AdamW(self.model.parameters(), lr=lr, eps=1e-8)
        self.loss_fn = torch.nn.BCEWithLogitsLoss()

        total_steps = len(train_loader) * epochs
        warmup_steps = int(0.05 * total_steps)

        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            warmup_steps,
            total_steps,
        )

        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

    def train(self):
        for epoch in range(self.epochs):
            print(f"\nEpoch {epoch + 1}/{self.epochs}")
            self.model.train()

            total_loss = 0
            total_acc = 0

            for batch in tqdm(self.train_loader):
                self.optimizer.zero_grad()

                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                logits = self.model(input_ids, attention_mask)
                loss = self.loss_fn(logits.squeeze(1), labels)

                loss.backward()
                self.optimizer.step()
                self.scheduler.step()

                total_loss += loss.item()

                preds = (torch.sigmoid(logits).squeeze(1) >= 0.5).detach().cpu().numpy()
                total_acc += accuracy_score(labels.detach().cpu().numpy(), preds)

            print(f"Train Loss: {total_loss / len(self.train_loader):.4f}")
            print(f"Train Acc: {total_acc / len(self.train_loader):.4f}")

            self.evaluate(epoch)

    def evaluate(self, epoch):
        self.model.eval()

        total_loss = 0
        total_acc = 0

        with torch.no_grad():
            for batch in self.val_loader:
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                logits = self.model(input_ids, attention_mask)
                loss = self.loss_fn(logits.squeeze(1), labels)

                total_loss += loss.item()

                preds = (torch.sigmoid(logits).squeeze(1) >= 0.5).detach().cpu().numpy()
                total_acc += accuracy_score(labels.detach().cpu().numpy(), preds)

        print(f"Val Loss: {total_loss / len(self.val_loader):.4f}")
        print(f"Val Acc: {total_acc / len(self.val_loader):.4f}")

        tokenizer.save_pretrained(self.output_dir)

        torch.save(
            self.model.state_dict(),
            self.output_dir / f"model_epoch_{epoch}.pt",
        )

In [11]:
# =========================
# 8. Train baseline verifier
# =========================

train_split, dev_split = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["Class"],
    random_state=RANDOM_STATE,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

train_dataset = TextDataset(train_split, tokenizer, MAX_LENGTH)
dev_dataset = TextDataset(dev_split, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE)

model = CustomClassifier(
    model_name=BASE_MODEL,
    freeze_base_layer=False,
)

print_trainable_parameters(model)

trainer = TrainerModule(
    model=model,
    train_loader=train_loader,
    val_loader=dev_loader,
    epochs=EPOCHS,
    lr=LR,
    output_dir=MODEL_DIR,
)

trainer.train()

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

C:\Users\aiger\Documents\2026SS\AIR\git_personal\AIR_Group_Task\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aiger\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Train Loss: 0.4509
Train Acc: 0.8084
Val Loss: 0.3947
Val Acc: 0.8411

Epoch 2/3


100%|██████████| 3144/3144 [2:27:19<00:00,  2.81s/it]  


Train Loss: 0.2902
Train Acc: 0.8768
Val Loss: 0.2680
Val Acc: 0.8911

Epoch 3/3


100%|██████████| 3144/3144 [2:13:30<00:00,  2.55s/it]  


Train Loss: 0.1656
Train Acc: 0.9319
Val Loss: 0.2435
Val Acc: 0.9119


In [10]:
# =========================
# 9. Prediction helper
# =========================

def remove_label_pattern(text):
    text = re.sub(
        r"(\[?\s*Justification\s*\]?:?\s*)|(\[Label\]:\s*(True|False|Conflicting))",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()
    return text.replace("\n", " ")

In [11]:
class VerifierEvaluator:
    def __init__(
        self,
        model_path,
        tokenizer_path,
        base_model,
        device="cuda",
    ):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")

        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = CustomClassifier(
            model_name=base_model,
            freeze_base_layer=False,
        )

        self.model.load_state_dict(torch.load(model_path, map_location=self.device))
        self.model.to(self.device)
        self.model.eval()

    def encode_input(self, claim, verdict, justification, max_length=150):
        text = f"Claim: {claim}\nVerdict: {verdict}\nJustification: {justification}"

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )

        return (
            encoding["input_ids"].to(self.device),
            encoding["attention_mask"].to(self.device),
        )

    def score(self, claim, verdict, justification):
        input_ids, attention_mask = self.encode_input(claim, verdict, justification)

        with torch.no_grad():
            return float(self.model(input_ids, attention_mask).item())

In [15]:
# =========================
# 10. Generate predictions
# =========================

BEST_EPOCH = EPOCHS - 1
MODEL_PATH = MODEL_DIR / f"model_epoch_{BEST_EPOCH}.pt"

with open(VAL_PATH, "r", encoding="utf-8") as f:
    val_data = json.load(f)

evaluator = VerifierEvaluator(
    model_path=MODEL_PATH,
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
)

predictions = []

for idx, sample in enumerate(tqdm(val_data)):
    verdict_list = []
    verifier_score_list = []
    justification_list = []

    for trace_idx in range(len(sample["Reasoning_traces"])):
        justification = remove_label_pattern(
            sample["Reasoning_traces"][trace_idx]
        ).split("Label:")[0]

        verdict = sample["Verdict_list"][trace_idx].lower()

        score = evaluator.score(
            claim=sample["claim"],
            verdict=verdict,
            justification=justification,
        )

        verdict_list.append(sample["Verdict_list"][trace_idx])
        justification_list.append(justification)
        verifier_score_list.append(score)

    best_idx = int(np.argmax(np.array(verifier_score_list)))
    best_verdict = verdict_list[best_idx]

    predictions.append(
        {
            "query_id": sample.get("query_id", idx),
            "Claim": sample["claim"],
            "Label": sample["label"],
            "Verdict_BoN": best_verdict,
            "BoN_Verdict_list": verdict_list,
            "Reasoning_traces": justification_list,
            "score_list": verifier_score_list,
        }
    )



Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 1600/1600 [25:44<00:00,  1.04it/s]


In [16]:
with open(PRED_PATH, "w", encoding="utf-8") as fp:
    json.dump(predictions, fp, indent=4, ensure_ascii=False)

print(f"Saved predictions to {PRED_PATH}")
print("Number of predictions:", len(predictions))

Saved predictions to output/RM_prediction/distilbert_predictions.json
Number of predictions: 1600


In [23]:
# =========================
# 11. Run provided scorer
# =========================

REPO_PRED_DIR = PROJECT_ROOT / "output" / "RM_prediction"
os.makedirs(REPO_PRED_DIR, exist_ok=True)

shutil.copy(
    PRED_PATH,
    REPO_PRED_DIR / "clef_predictions.json"
)

subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "task2" / "scorer.py")],
    check=True,
    cwd=str(PROJECT_ROOT),
)

shutil.copy(
    REPO_PRED_DIR / "result.csv",
    RESULT_DIR / "result.csv"
)

shutil.copy(
    REPO_PRED_DIR / "per_sample_ir.csv",
    RESULT_DIR / "per_sample_ir.csv"
)


CompletedProcess(args=['C:\\Users\\aiger\\Documents\\2026SS\\AIR\\git_personal\\AIR_Group_Task\\.venv\\Scripts\\python.exe', 'task2/scorer.py'], returncode=0)